# M-BLiMP (Hindi) for bilingual en-hi GPT-BERT, on Colab

Downloads the `_ema.bin` checkpoints from `pulipakav-1/en-hi-gptbert-checkpoints` (raw upload, no HF model format), shallow-clones the repo for the model code / config / tokenizer, and runs the same pseudo-log-likelihood M-BLiMP scoring as `eval_gptbert_bilingual_mblimp.py`.

In [ ]:
!pip install -q transformers datasets torch tqdm huggingface_hub

In [ ]:
# Shallow-clone the repo to get modeling_gptbert.py / configuration_gptbert.py / the config json / the tokenizer json.
# (Only the code + small config/tokenizer files are needed from git -- the large checkpoints are pulled from HF below.)
!git clone --depth 1 https://github.com/vishnup22/BabyLM.git
%cd BabyLM

In [ ]:
import sys
import json
from pathlib import Path

import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import PreTrainedTokenizerFast
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "gpt-bert"))
from configuration_gptbert import GptBertConfig
from modeling_gptbert import GptBertForMaskedLM

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────────
CHECKPOINT_REPO = 'pulipakav-1/en-hi-gptbert-checkpoints'
SEEDS   = [1, 2]
VARIANT = 'ema'   # matches the *_ema.bin files uploaded

CONFIG_PATH = REPO_ROOT / 'Babylm2026/eng-hin/gptbert multi/configs/multilingual.json'
# tokenizer JSON isn't in git (build artifact) -- pulled from the checkpoint repo below instead
TOKENIZER_FILENAME = 'tokenizer_en_hi_vs32768.json'
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
@torch.no_grad()
def score_gptbert_pll(model, token_ids, cls_id, mask_id):
    """Pseudo-log-likelihood, same convention as monolingual eval/hindi.py::score_gptbert_pll."""
    full_ids = [cls_id] + token_ids
    seq_len  = len(full_ids)
    n        = seq_len - 1
    if n == 0:
        return float('-inf')
    base     = torch.tensor(full_ids, dtype=torch.long, device=DEVICE)
    input_t  = base.unsqueeze(1).expand(-1, n).clone()
    labels_t = torch.full((seq_len, n), -100, dtype=torch.long, device=DEVICE)
    for j in range(n):
        pos = j + 1
        input_t[pos, j]  = mask_id
        labels_t[pos, j] = base[pos]
    attn = torch.zeros(n, 1, seq_len, seq_len, dtype=torch.bool, device=DEVICE)
    static_emb, rel_emb = model.embedding(input_t)
    hidden               = model.transformer(static_emb, attn, rel_emb)
    logits               = model.classifier(hidden, labels_t)
    log_probs = F.log_softmax(logits, dim=-1)
    gold      = base[1:].to(DEVICE)
    return log_probs[torch.arange(n, device=DEVICE), gold].sum().item()


PLL_MAX_LEN = 128  # score_gptbert_pll is O(seq_len^2) memory -- cap sequence length before calling it


def eval_mblimp(model, tokenizer, cls_id, mask_id):
    ds = load_dataset('jumelet/multiblimp', 'hin', split='train')
    correct = 0
    for row in tqdm(ds, desc='  M-BLiMP', leave=False):
        good_ids = tokenizer.encode(row['sen'], add_special_tokens=False)[:PLL_MAX_LEN]
        bad_ids  = tokenizer.encode(row['wrong_sen'], add_special_tokens=False)[:PLL_MAX_LEN]
        good = score_gptbert_pll(model, good_ids, cls_id, mask_id)
        bad  = score_gptbert_pll(model, bad_ids, cls_id, mask_id)
        if good > bad:
            correct += 1
    return correct / len(ds)

In [ ]:
with open(CONFIG_PATH) as f:
    cfg_dict = json.load(f)
config = GptBertConfig(**cfg_dict)

tokenizer_path = hf_hub_download(repo_id=CHECKPOINT_REPO, filename=TOKENIZER_FILENAME)
tokenizer = PreTrainedTokenizerFast(tokenizer_file=str(tokenizer_path))
cls_id  = tokenizer.convert_tokens_to_ids('<s>')
mask_id = tokenizer.convert_tokens_to_ids('<mask>')
print(f'cls_id={cls_id}  mask_id={mask_id}')

In [ ]:
results = {}

for seed in SEEDS:
    filename = f'en-hi-gptbert-seed{seed}_{VARIANT}.bin' if VARIANT != 'plain' else f'en-hi-gptbert-seed{seed}.bin'
    print(f'\n=== seed {seed} ({filename}) ===')

    ckpt_path = hf_hub_download(repo_id=CHECKPOINT_REPO, filename=filename)
    state_dict = torch.load(ckpt_path, map_location='cpu')

    model = GptBertForMaskedLM(config)
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    if missing:
        print(f'  WARNING - missing keys: {missing}')
    if unexpected:
        print(f'  WARNING - unexpected keys: {unexpected}')
    model = model.eval().to(DEVICE)

    acc = eval_mblimp(model, tokenizer, cls_id, mask_id)
    results[seed] = round(acc, 4)
    print(f'  M-BLiMP (hin): {acc:.4f}')

    del model
    torch.cuda.empty_cache()

In [ ]:
print('\n=== M-BLiMP Results (en-hi-gptbert) ===')
for seed, acc in results.items():
    print(f'  seed {seed}: {acc}')
if results:
    mean_acc = sum(results.values()) / len(results)
    print(f'  mean:   {mean_acc:.4f}')

## BLiMP (English, 67 tasks) for en-hi-gptbert seed1

Same pseudo-log-likelihood scoring, run against `BabyLM-community/BabyLM-BLIMP-Filtered` instead of M-BLiMP.

In [ ]:
BLIMP_TASKS = [
    "adjunct_island", "anaphor_gender_agreement", "anaphor_number_agreement",
    "animate_subject_passive", "animate_subject_trans", "causative",
    "complex_NP_island",
    "coordinate_structure_constraint_complex_left_branch",
    "coordinate_structure_constraint_object_extraction",
    "determiner_noun_agreement_1", "determiner_noun_agreement_2",
    "determiner_noun_agreement_irregular_1", "determiner_noun_agreement_irregular_2",
    "determiner_noun_agreement_with_adj_2",
    "determiner_noun_agreement_with_adj_irregular_1",
    "determiner_noun_agreement_with_adj_irregular_2",
    "determiner_noun_agreement_with_adjective_1",
    "distractor_agreement_relational_noun", "distractor_agreement_relative_clause",
    "drop_argument", "ellipsis_n_bar_1", "ellipsis_n_bar_2",
    "existential_there_object_raising",
    "existential_there_quantifiers_1", "existential_there_quantifiers_2",
    "existential_there_subject_raising", "expletive_it_object_raising",
    "inchoative", "intransitive",
    "irregular_past_participle_adjectives", "irregular_past_participle_verbs",
    "irregular_plural_subject_verb_agreement_1",
    "irregular_plural_subject_verb_agreement_2",
    "left_branch_island_echo_question", "left_branch_island_simple_question",
    "matrix_question_npi_licensor_present",
    "npi_present_1", "npi_present_2",
    "only_npi_licensor_present", "only_npi_scope",
    "passive_1", "passive_2",
    "principle_A_c_command", "principle_A_case_1", "principle_A_case_2",
    "principle_A_domain_1", "principle_A_domain_2", "principle_A_domain_3",
    "principle_A_reconstruction",
    "regular_plural_subject_verb_agreement_1", "regular_plural_subject_verb_agreement_2",
    "sentential_negation_npi_licensor_present", "sentential_negation_npi_scope",
    "sentential_subject_island",
    "superlative_quantifiers_1", "superlative_quantifiers_2",
    "tough_vs_raising_1", "tough_vs_raising_2", "transitive", "wh_island",
    "wh_questions_object_gap", "wh_questions_subject_gap",
    "wh_questions_subject_gap_long_distance",
    "wh_vs_that_no_gap", "wh_vs_that_no_gap_long_distance",
    "wh_vs_that_with_gap", "wh_vs_that_with_gap_long_distance",
]


def eval_blimp(model, tokenizer, cls_id, mask_id):
    results = {}
    for task in tqdm(BLIMP_TASKS, desc='BLiMP'):
        ds = load_dataset('BabyLM-community/BabyLM-BLIMP-Filtered', task, split='train')
        correct = 0
        for row in ds:
            good_ids = tokenizer.encode(row['sentence_good'], add_special_tokens=False)[:PLL_MAX_LEN]
            bad_ids  = tokenizer.encode(row['sentence_bad'], add_special_tokens=False)[:PLL_MAX_LEN]
            good = score_gptbert_pll(model, good_ids, cls_id, mask_id)
            bad  = score_gptbert_pll(model, bad_ids, cls_id, mask_id)
            if good > bad:
                correct += 1
        acc = correct / len(ds)
        results[task] = round(acc, 4)
        tqdm.write(f'  {task:<60s} {acc:.4f}')
    macro = sum(results.values()) / len(results)
    results['macro_avg'] = round(macro, 4)
    print(f'BLiMP macro avg: {macro:.4f}')
    return results

In [ ]:
seed1_filename = f'en-hi-gptbert-seed1_{VARIANT}.bin' if VARIANT != 'plain' else 'en-hi-gptbert-seed1.bin'
ckpt_path = hf_hub_download(repo_id=CHECKPOINT_REPO, filename=seed1_filename)
state_dict = torch.load(ckpt_path, map_location='cpu')

model = GptBertForMaskedLM(config)
missing, unexpected = model.load_state_dict(state_dict, strict=True)
if missing:
    print(f'  WARNING - missing keys: {missing}')
if unexpected:
    print(f'  WARNING - unexpected keys: {unexpected}')
model = model.eval().to(DEVICE)

blimp_results = eval_blimp(model, tokenizer, cls_id, mask_id)

del model
torch.cuda.empty_cache()

In [ ]:
print('\n=== BLiMP Results (en-hi-gptbert seed1) ===')
print(f"macro_avg: {blimp_results['macro_avg']}")

## Perplexity (causal, correct), SIB-200, MuBench (English) for en-hi-gptbert seed1

Perplexity uses the causal-mode method (lower-triangular mask + shifted targets), same as `eval_gptbert_causal.ipynb` -- comparable to GPT-2, not the PLL pseudo-perplexity. SIB-200 and MuBench use pseudo-log-likelihood scoring (length-normalised), same as `monolingual eval/english.py`.

In [ ]:
MUBENCH_DATASET_ID = 'aialt/MuBench'

MUBENCH_TASKS = [
    'ARCChallengeDataset_local_template_en',
    'ARCEasyDataset_local_template_en',
    'BMLAMADataset_local_template_en',
    'GPQADataset_local_template_en',
    'HellaswagDataset_local_template_en',
    'MMLUDataset_local_template_en',
    'MMLUProDataset_local_template_en',
    'MNLIDataset_local_template_en',
    'SNLIDataset_local_template_en',
    'StoryClozeDataset_local_template_en',
    'TruthfulQADataset_local_template_en',
    'WinoGrandeDataset_local_template_en',
]

SIB200_CONFIG = 'eng_Latn'
SIB200_LABELS = [
    'science/technology', 'travel', 'politics', 'sports',
    'health', 'entertainment', 'geography',
]

PLL_MAX_LEN = 128  # score_gptbert_pll is O(seq_len^2) memory -- long MuBench prompts (e.g. GPQA) OOM without this


def eval_sib200(model, tokenizer, cls_id, mask_id):
    ds = load_dataset('Davlan/sib200', SIB200_CONFIG, split='test')
    correct = 0
    for row in tqdm(ds, desc='SIB-200', leave=False):
        text = row['text']
        best_score, best_label = float('-inf'), None
        for label in SIB200_LABELS:
            prompt = f'{text}\nTopic: {label}'
            ids = tokenizer.encode(prompt, add_special_tokens=False)[:PLL_MAX_LEN]
            score = score_gptbert_pll(model, ids, cls_id, mask_id) / max(len(ids), 1)
            if score > best_score:
                best_score, best_label = score, label
        if best_label == row['category']:
            correct += 1
    acc = correct / len(ds)
    print(f'SIB-200 [{SIB200_CONFIG}]: {acc:.4f}')
    return round(acc, 4)


def eval_mubench(model, tokenizer, cls_id, mask_id):
    results = {}
    for task_config in tqdm(MUBENCH_TASKS, desc='MuBench'):
        task_key = task_config.replace('Dataset_local_template_en', '')
        try:
            ds = load_dataset(MUBENCH_DATASET_ID, task_config, split='test')
        except Exception as e:
            tqdm.write(f'  SKIP {task_config}: {e}')
            continue
        correct = 0
        for row in ds:
            prompt  = row['prompt']
            choices = row['choices']
            label   = row['label']
            scores = []
            for choice in choices:
                text = prompt + choice
                ids  = tokenizer.encode(text, add_special_tokens=False)[:PLL_MAX_LEN]
                s    = score_gptbert_pll(model, ids, cls_id, mask_id) / max(len(ids), 1)
                scores.append(s)
            if scores.index(max(scores)) == label:
                correct += 1
        acc = correct / len(ds)
        results[task_key] = round(acc, 4)
        tqdm.write(f'  {task_key:<30s} {acc:.4f}')
    if results:
        results['avg'] = round(sum(results.values()) / len(results), 4)
    return results

In [ ]:
def compute_causal_perplexity(model, tokenizer, texts, cls_id, pad_id, max_seq_len=128, batch_size=32):
    model.eval()
    total_nll, total_tokens = 0.0, 0

    for i in tqdm(range(0, len(texts), batch_size), leave=False):
        batch_texts = texts[i:i + batch_size]
        token_lists = [tokenizer.encode(t, add_special_tokens=False)[:max_seq_len - 1] for t in batch_texts]
        token_lists = [t for t in token_lists if len(t) > 0]
        if not token_lists:
            continue
        B = len(token_lists)
        L = max(len(t) for t in token_lists) + 1

        input_ids = torch.full((L, B), pad_id, dtype=torch.long)
        labels    = torch.full((L, B), -100,   dtype=torch.long)
        valid_lens = torch.zeros(B, dtype=torch.long)

        for b, tok in enumerate(token_lists):
            n = len(tok)
            seq = torch.tensor([cls_id] + tok, dtype=torch.long)
            tgt = torch.tensor(tok + [-100],   dtype=torch.long)
            input_ids[:n + 1, b] = seq
            labels[:n + 1, b]    = tgt
            valid_lens[b] = n + 1

        input_ids = input_ids.to(DEVICE)
        labels    = labels.to(DEVICE)
        valid_lens = valid_lens.to(DEVICE)

        causal_block = torch.triu(torch.ones(L, L, dtype=torch.bool, device=DEVICE), diagonal=1)
        pad_block = torch.arange(L, device=DEVICE).unsqueeze(0) >= valid_lens.unsqueeze(1)
        pad_block = pad_block.unsqueeze(1).expand(-1, L, -1)
        mask_out = (causal_block.unsqueeze(0) | pad_block).unsqueeze(1)

        with torch.no_grad():
            static_emb, rel_emb = model.embedding(input_ids)
            hidden = model.transformer(static_emb, mask_out, rel_emb)
            logits = model.classifier(hidden, labels)

        gold = labels.flatten()
        gold = gold[gold != -100]
        if gold.numel() == 0:
            continue
        nll_sum = F.cross_entropy(logits, gold, reduction='sum').item()
        total_nll    += nll_sum
        total_tokens += gold.numel()

    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float('inf')


def load_english_test_texts():
    ds = load_dataset('text', data_files={'test': 'hf://datasets/BabyLM-community/BabyLM-Test/*.test'}, split='test')
    return [row['text'] for row in ds if row.get('text')]


def load_english_os_texts():
    ds = load_dataset('Helsinki-NLP/opus-100', 'en-hi', split='train', streaming=True)
    texts = []
    for row in ds:
        text = row['translation']['en'].strip()
        if text:
            texts.append(text)
    return texts

In [ ]:
pad_id = tokenizer.convert_tokens_to_ids('<pad>')
if pad_id is None or pad_id < 0:
    pad_id = 0

seed1_filename = f'en-hi-gptbert-seed1_{VARIANT}.bin' if VARIANT != 'plain' else 'en-hi-gptbert-seed1.bin'
ckpt_path = hf_hub_download(repo_id=CHECKPOINT_REPO, filename=seed1_filename)
state_dict = torch.load(ckpt_path, map_location='cpu')

model = GptBertForMaskedLM(config)
missing, unexpected = model.load_state_dict(state_dict, strict=True)
if missing:
    print(f'  WARNING - missing keys: {missing}')
if unexpected:
    print(f'  WARNING - unexpected keys: {unexpected}')
model = model.eval().to(DEVICE)

print('Loading English test/OS-data ...')
en_test_texts = load_english_test_texts()
en_os_texts   = load_english_os_texts()
print(f'  test: {len(en_test_texts):,} lines,  OS-data: {len(en_os_texts):,} lines')

causal_test = compute_causal_perplexity(model, tokenizer, en_test_texts, cls_id, pad_id)
causal_os   = compute_causal_perplexity(model, tokenizer, en_os_texts, cls_id, pad_id)
sib200_acc  = eval_sib200(model, tokenizer, cls_id, mask_id)
mubench_res = eval_mubench(model, tokenizer, cls_id, mask_id)

print(f'\nCausal Test PPL: {causal_test:.4f}')
print(f'Causal OS-data PPL: {causal_os:.4f}')
print(f'SIB-200: {sib200_acc}')
print(f"MuBench avg: {mubench_res.get('avg')}")

del model
torch.cuda.empty_cache()